In [454]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


In [455]:
accounts = pd.read_csv('ravenstack_accounts.csv')
churn_events = pd.read_csv('ravenstack_churn_events.csv')
feature_usage = pd.read_csv('ravenstack_feature_usage.csv')
subscription = pd.read_csv('ravenstack_subscriptions.csv')
support = pd.read_csv('ravenstack_support_tickets.csv')

In [456]:
accounts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   account_id       500 non-null    object
 1   account_name     500 non-null    object
 2   industry         500 non-null    object
 3   country          500 non-null    object
 4   signup_date      500 non-null    object
 5   referral_source  500 non-null    object
 6   plan_tier        500 non-null    object
 7   seats            500 non-null    int64 
 8   is_trial         500 non-null    bool  
 9   churn_flag       500 non-null    bool  
dtypes: bool(2), int64(1), object(7)
memory usage: 32.4+ KB


In [457]:
churn_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   churn_event_id            600 non-null    object 
 1   account_id                600 non-null    object 
 2   churn_date                600 non-null    object 
 3   reason_code               600 non-null    object 
 4   refund_amount_usd         600 non-null    float64
 5   preceding_upgrade_flag    600 non-null    bool   
 6   preceding_downgrade_flag  600 non-null    bool   
 7   is_reactivation           600 non-null    bool   
 8   feedback_text             452 non-null    object 
dtypes: bool(3), float64(1), object(5)
memory usage: 30.0+ KB


In [458]:
feature_usage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   usage_id             25000 non-null  object
 1   subscription_id      25000 non-null  object
 2   usage_date           25000 non-null  object
 3   feature_name         25000 non-null  object
 4   usage_count          25000 non-null  int64 
 5   usage_duration_secs  25000 non-null  int64 
 6   error_count          25000 non-null  int64 
 7   is_beta_feature      25000 non-null  bool  
dtypes: bool(1), int64(3), object(4)
memory usage: 1.4+ MB


In [459]:
subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   subscription_id    5000 non-null   object
 1   account_id         5000 non-null   object
 2   start_date         5000 non-null   object
 3   end_date           486 non-null    object
 4   plan_tier          5000 non-null   object
 5   seats              5000 non-null   int64 
 6   mrr_amount         5000 non-null   int64 
 7   arr_amount         5000 non-null   int64 
 8   is_trial           5000 non-null   bool  
 9   upgrade_flag       5000 non-null   bool  
 10  downgrade_flag     5000 non-null   bool  
 11  churn_flag         5000 non-null   bool  
 12  billing_frequency  5000 non-null   object
 13  auto_renew_flag    5000 non-null   bool  
dtypes: bool(5), int64(3), object(6)
memory usage: 376.1+ KB


In [460]:
support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ticket_id                    2000 non-null   object 
 1   account_id                   2000 non-null   object 
 2   submitted_at                 2000 non-null   object 
 3   closed_at                    2000 non-null   object 
 4   resolution_time_hours        2000 non-null   float64
 5   priority                     2000 non-null   object 
 6   first_response_time_minutes  2000 non-null   int64  
 7   satisfaction_score           1175 non-null   float64
 8   escalation_flag              2000 non-null   bool   
dtypes: bool(1), float64(2), int64(1), object(5)
memory usage: 127.1+ KB


In [461]:
accounts['industry'].unique()

array(['EdTech', 'FinTech', 'DevTools', 'HealthTech', 'Cybersecurity'],
      dtype=object)

In [462]:
accounts['country'].unique()

array(['US', 'IN', 'UK', 'CA', 'DE', 'AU', 'FR'], dtype=object)

In [463]:
accounts['referral_source'].unique()

array(['partner', 'other', 'organic', 'event', 'ads'], dtype=object)

In [464]:
accounts['plan_tier'].unique()

array(['Basic', 'Enterprise', 'Pro'], dtype=object)

In [465]:
churn_events.sample(2)

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
488,C-06b32c,A-2ebb4b,2023-08-19,pricing,0.0,False,False,False,missing features
303,C-d2077d,A-cab532,2024-07-10,support,0.0,True,False,False,too expensive


In [466]:
churn_events['churn_date'] = pd.to_datetime(churn_events['churn_date'])

In [467]:
churn_events_cpy = pd.get_dummies(churn_events, columns=['reason_code', 'feedback_text'], dtype=int)
churn_events_cpy.drop(columns=['churn_event_id'], inplace=True)
churn_events_cpy.sample(2)

,account_id,churn_date,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,reason_code_budget,reason_code_competitor,reason_code_features,reason_code_pricing,reason_code_support,reason_code_unknown,feedback_text_missing features,feedback_text_switched to competitor,feedback_text_too expensive
370,A-e08cd3,2024-12-17,0.0,False,False,False,0,0,1,0,0,0,0,0,1
413,A-f03140,2024-10-03,0.0,False,False,False,0,1,0,0,0,0,1,0,0


In [468]:
churn_events_cpy.rename(columns={
    'feedback_text_missing features': 'feedback_text_missing_features',
    'feedback_text_switched to competitor': 'feedback_text_switched_to_competitor',
    'feedback_text_too expensive': 'feedback_text_too_expensive'
}, inplace=True)

In [469]:
churn_events_cpy.columns

Index(['account_id', 'churn_date', 'refund_amount_usd',
       'preceding_upgrade_flag', 'preceding_downgrade_flag', 'is_reactivation',
       'reason_code_budget', 'reason_code_competitor', 'reason_code_features',
       'reason_code_pricing', 'reason_code_support', 'reason_code_unknown',
       'feedback_text_missing_features',
       'feedback_text_switched_to_competitor', 'feedback_text_too_expensive'],
      dtype='object')

In [470]:
churn_events_cpy

,account_id,churn_date,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,reason_code_budget,reason_code_competitor,reason_code_features,reason_code_pricing,reason_code_support,reason_code_unknown,feedback_text_missing_features,feedback_text_switched_to_competitor,feedback_text_too_expensive
0,A-c37cab,2024-10-27,4.03,False,False,False,0,0,0,1,0,0,0,1,0
1,A-37f969,2024-06-25,96.45,True,False,False,0,0,0,0,1,0,0,0,0
2,A-b07346,2024-11-12,0.00,False,False,False,1,0,0,0,0,0,1,0,0
3,A-1e50e0,2023-11-01,54.94,False,False,False,1,0,0,0,0,0,0,1,0
4,A-956988,2024-12-30,0.00,False,True,True,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,A-702032,2024-06-14,0.00,False,False,False,0,1,0,0,0,0,0,1,0
596,A-dbc825,2024-02-03,0.00,False,False,False,1,0,0,0,0,0,0,0,0
597,A-0a282f,2024-12-31,62.66,False,True,False,0,0,0,0,1,0,0,0,1
598,A-e5d6ab,2024-05-11,0.00,True,False,False,0,1,0,0,0,0,0,0,0


In [471]:
churn_events_cpy = churn_events_cpy.groupby('account_id').agg({
    'churn_date': 'max',
    'refund_amount_usd': 'sum',
    'preceding_upgrade_flag': 'sum',
    'preceding_downgrade_flag': 'sum',
    'is_reactivation' : 'sum',
    'reason_code_budget': 'sum',
    'reason_code_competitor': 'sum',
    'reason_code_features': 'sum',
    'reason_code_pricing': 'sum',
    'reason_code_support': 'sum',
    'reason_code_unknown': 'sum',
    'feedback_text_missing_features': 'sum',
    'feedback_text_switched_to_competitor': 'sum',
    'feedback_text_too_expensive': 'sum'}).reset_index()

In [472]:
final_df = accounts.merge(churn_events_cpy, on='account_id', how='left')

In [473]:
final_df['account_id'].nunique()

500

In [474]:
final_df

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,...,is_reactivation,reason_code_budget,reason_code_competitor,reason_code_features,reason_code_pricing,reason_code_support,reason_code_unknown,feedback_text_missing_features,feedback_text_switched_to_competitor,feedback_text_too_expensive
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,A-8ae3fc,Company_495,DevTools,CA,2024-06-28,ads,Pro,9,False,False,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
496,A-55f257,Company_496,FinTech,US,2023-12-21,organic,Basic,9,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497,A-d26ab4,Company_497,DevTools,UK,2024-11-07,organic,Basic,9,False,True,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
498,A-712533,Company_498,EdTech,US,2023-07-31,organic,Pro,18,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


In [475]:
feature_usage.sample(2)

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
10000,U-646c31,S-5163bd,2023-01-23,feature_1,15,5865,0,False
6306,U-e39412,S-22cbb3,2024-02-23,feature_9,9,4365,0,False


In [476]:
subscription.sample(2)

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
2101,S-41acf0,A-378b99,2024-11-05,NaN,Enterprise,16,3184,38208,False,False,False,False,annual,True
3265,S-51569a,A-e4bf56,2024-12-25,NaN,Enterprise,22,4378,52536,False,False,False,False,monthly,True


In [477]:
subscription['subscription_id'].nunique()

5000

In [478]:
feature_usage['subscription_id'].nunique()

4967

In [479]:
feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'])

In [480]:
feature_usage_cpy = pd.get_dummies(feature_usage, columns=['feature_name'], dtype=int)
feature_usage_cpy.drop(columns=['usage_id'], inplace=True)
feature_usage_cpy.sample(2)

,subscription_id,usage_date,usage_count,usage_duration_secs,error_count,is_beta_feature,feature_name_feature_1,feature_name_feature_10,feature_name_feature_11,feature_name_feature_12,...,feature_name_feature_37,feature_name_feature_38,feature_name_feature_39,feature_name_feature_4,feature_name_feature_40,feature_name_feature_5,feature_name_feature_6,feature_name_feature_7,feature_name_feature_8,feature_name_feature_9
16974,S-8932ce,2024-11-28,9,5157,3,False,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1918,S-ba8f61,2023-01-16,10,730,1,False,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [481]:
cols_to_sum = feature_usage_cpy.columns.difference(['subscription_id', 'usage_date'])
feature_usage_cpy = feature_usage_cpy.groupby('subscription_id').agg({
    'usage_date': 'max',
    **{col: 'sum' for col in cols_to_sum}
}).reset_index()

In [482]:
subscription = subscription.merge(feature_usage_cpy, on='subscription_id', how='left')

In [483]:
subscription['billing_frequency']

,billing_frequency
0,monthly
1,monthly
2,annual
3,monthly
4,monthly
...,...
4995,monthly
4996,monthly
4997,annual
4998,monthly


In [484]:
subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 59 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   subscription_id          5000 non-null   object        
 1   account_id               5000 non-null   object        
 2   start_date               5000 non-null   object        
 3   end_date                 486 non-null    object        
 4   plan_tier                5000 non-null   object        
 5   seats                    5000 non-null   int64         
 6   mrr_amount               5000 non-null   int64         
 7   arr_amount               5000 non-null   int64         
 8   is_trial                 5000 non-null   bool          
 9   upgrade_flag             5000 non-null   bool          
 10  downgrade_flag           5000 non-null   bool          
 11  churn_flag               5000 non-null   bool          
 12  billing_frequency        5000 non-

In [485]:
subscription['start_date'] = pd.to_datetime(subscription['start_date'])
subscription['end_date'] = pd.to_datetime(subscription['end_date'])

In [486]:
subscription_cpy = pd.get_dummies(subscription, columns=['plan_tier', 'billing_frequency'], dtype=int)
subscription_cpy.drop(columns=['subscription_id'], inplace=True)
subscription_cpy.sample(2)

,account_id,start_date,end_date,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,...,feature_name_feature_8,feature_name_feature_9,is_beta_feature,usage_count,usage_duration_secs,plan_tier_Basic,plan_tier_Enterprise,plan_tier_Pro,billing_frequency_annual,billing_frequency_monthly
3986,A-a0ca4e,2024-05-15,NaT,11,209,2508,False,False,False,False,...,0.0,0.0,0.0,44.0,18229.0,1,0,0,1,0
3143,A-0dbdc6,2024-10-25,NaT,30,5970,71640,False,False,False,False,...,1.0,0.0,1.0,50.0,11597.0,0,1,0,0,1


In [487]:
subscription_cpy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 61 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   account_id                 5000 non-null   object        
 1   start_date                 5000 non-null   datetime64[ns]
 2   end_date                   486 non-null    datetime64[ns]
 3   seats                      5000 non-null   int64         
 4   mrr_amount                 5000 non-null   int64         
 5   arr_amount                 5000 non-null   int64         
 6   is_trial                   5000 non-null   bool          
 7   upgrade_flag               5000 non-null   bool          
 8   downgrade_flag             5000 non-null   bool          
 9   churn_flag                 5000 non-null   bool          
 10  auto_renew_flag            5000 non-null   bool          
 11  usage_date                 4967 non-null   datetime64[ns]
 12  error_

In [488]:
col_to_sum = subscription_cpy.columns.difference(['account_id', 'start_date', 'end_date', 'usage_date'])
subscription_cpy = subscription_cpy.groupby('account_id').agg({
    'start_date': 'min',
    'end_date': 'max',
    'usage_date': 'max',
    **{col: 'sum' for col in col_to_sum}
}).reset_index()



In [489]:
final_df = final_df.merge(subscription_cpy, on='account_id', how='left')

In [490]:
final_df

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats_x,is_trial_x,churn_flag_x,...,is_beta_feature,is_trial_y,mrr_amount,plan_tier_Basic,plan_tier_Enterprise,plan_tier_Pro,seats_y,upgrade_flag,usage_count,usage_duration_secs
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,...,4.0,2,12603,3,2,5,312,1,535.0,152339.0
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,...,2.0,0,10004,4,1,3,176,3,355.0,101136.0
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,...,5.0,1,18286,4,4,7,282,1,821.0,251210.0
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False,...,5.0,1,9275,3,1,3,209,2,382.0,102528.0
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True,...,4.0,1,48761,0,4,5,424,2,579.0,215779.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,A-8ae3fc,Company_495,DevTools,CA,2024-06-28,ads,Pro,9,False,False,...,5.0,1,4447,4,1,3,221,0,402.0,130242.0
496,A-55f257,Company_496,FinTech,US,2023-12-21,organic,Basic,9,False,False,...,6.0,3,12137,2,3,5,188,3,476.0,145972.0
497,A-d26ab4,Company_497,DevTools,UK,2024-11-07,organic,Basic,9,False,True,...,3.0,4,16613,2,4,3,202,0,458.0,121379.0
498,A-712533,Company_498,EdTech,US,2023-07-31,organic,Pro,18,False,False,...,4.0,2,19382,4,4,4,341,1,571.0,180056.0


In [491]:
support.sample(2)

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
778,T-8132dc,A-474ad4,2023-09-13,2023-09-14 09:00:00,33.0,urgent,11,NaN,False
1483,T-3b516f,A-1b7577,2023-05-22,2023-05-22 19:00:00,19.0,urgent,167,5.0,False


In [492]:
support['account_id'].nunique()

492

In [493]:
support['submitted_at'] = pd.to_datetime(support['submitted_at'])
support['closed_at'] = pd.to_datetime(support['closed_at'])


In [494]:
support_cpy = pd.get_dummies(support, columns=['priority'], dtype=int)
support_cpy.drop(columns=['ticket_id'], inplace=True)
support_cpy.sample(2)

,account_id,submitted_at,closed_at,resolution_time_hours,first_response_time_minutes,satisfaction_score,escalation_flag,priority_high,priority_low,priority_medium,priority_urgent
1935,A-180abf,2023-08-15,2023-08-17 17:00:00,65.0,138,NaN,False,0,0,1,0
344,A-812c5b,2023-02-20,2023-02-21 19:00:00,43.0,127,4.0,False,0,0,0,1


In [495]:
support_cpy = support_cpy.groupby('account_id').agg({
    'submitted_at': ['min', 'max'],
    'closed_at': ['min', 'max'],
    'resolution_time_hours': 'mean',
    'first_response_time_minutes': 'mean',
    'satisfaction_score': 'mean',
    'escalation_flag': 'sum',
    'priority_urgent': 'sum',
    'priority_high': 'sum',
    'priority_low': 'sum',
    'priority_medium': 'sum'})

In [496]:
support_cpy.columns = [
    '_'.join(col).strip() if isinstance(col, tuple) else col
    for col in support_cpy.columns
]

In [497]:
final_df = final_df.merge(support_cpy, on='account_id', how='left')

In [498]:
final_df.columns

Index(['account_id', 'account_name', 'industry', 'country', 'signup_date',
       'referral_source', 'plan_tier', 'seats_x', 'is_trial_x', 'churn_flag_x',
       'churn_date', 'refund_amount_usd', 'preceding_upgrade_flag',
       'preceding_downgrade_flag', 'is_reactivation', 'reason_code_budget',
       'reason_code_competitor', 'reason_code_features', 'reason_code_pricing',
       'reason_code_support', 'reason_code_unknown',
       'feedback_text_missing_features',
       'feedback_text_switched_to_competitor', 'feedback_text_too_expensive',
       'start_date', 'end_date', 'usage_date', 'arr_amount', 'auto_renew_flag',
       'billing_frequency_annual', 'billing_frequency_monthly', 'churn_flag_y',
       'downgrade_flag', 'error_count', 'feature_name_feature_1',
       'feature_name_feature_10', 'feature_name_feature_11',
       'feature_name_feature_12', 'feature_name_feature_13',
       'feature_name_feature_14', 'feature_name_feature_15',
       'feature_name_feature_16', 'featu

In [499]:
final_df.to_csv('final_SaaS_Churning_dataset')

In [500]:
final_df.columns

Index(['account_id', 'account_name', 'industry', 'country', 'signup_date',
       'referral_source', 'plan_tier', 'seats_x', 'is_trial_x', 'churn_flag_x',
       'churn_date', 'refund_amount_usd', 'preceding_upgrade_flag',
       'preceding_downgrade_flag', 'is_reactivation', 'reason_code_budget',
       'reason_code_competitor', 'reason_code_features', 'reason_code_pricing',
       'reason_code_support', 'reason_code_unknown',
       'feedback_text_missing_features',
       'feedback_text_switched_to_competitor', 'feedback_text_too_expensive',
       'start_date', 'end_date', 'usage_date', 'arr_amount', 'auto_renew_flag',
       'billing_frequency_annual', 'billing_frequency_monthly', 'churn_flag_y',
       'downgrade_flag', 'error_count', 'feature_name_feature_1',
       'feature_name_feature_10', 'feature_name_feature_11',
       'feature_name_feature_12', 'feature_name_feature_13',
       'feature_name_feature_14', 'feature_name_feature_15',
       'feature_name_feature_16', 'featu

In [501]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 96 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   account_id                            500 non-null    object        
 1   account_name                          500 non-null    object        
 2   industry                              500 non-null    object        
 3   country                               500 non-null    object        
 4   signup_date                           500 non-null    object        
 5   referral_source                       500 non-null    object        
 6   plan_tier                             500 non-null    object        
 7   seats_x                               500 non-null    int64         
 8   is_trial_x                            500 non-null    bool          
 9   churn_flag_x                          500 non-null    bool          
 10  ch

In [502]:
(final_df['churn_flag_x'] == final_df['churn_flag_y']).all()

np.False_

In [503]:
(final_df['seats_x'] == final_df['seats_y']).all()

np.False_

In [504]:
(final_df['is_trial_x'] == final_df['is_trial_y']).all()

np.False_

In [505]:
(final_df['churn_flag_x'] == final_df['churn_flag_y']).all()

np.False_

In [506]:
subscription['churn_flag'].value_counts()

,count
churn_flag,
False,4514
True,486


In [507]:
final_df['churn_flag_x']

,churn_flag_x
0,False
1,True
2,False
3,False
4,True
...,...
495,False
496,False
497,True
498,False


In [508]:
final_df['churn_flag_y']

,churn_flag_y
0,0
1,0
2,3
3,0
4,2
...,...
495,0
496,0
497,1
498,0


In [509]:
final_df.rename(columns={
    'churn_flag_x': 'churn_flag',
    'churn_flag_y': 'past_churn_count',
    'is_trial_x': 'is_trial_current',
    'is_trial_y': 'trial_count'
}, inplace=True)

In [510]:
final_df.drop(columns=['seats_y'], inplace=True)

In [511]:
final_df.describe()

,seats_x,churn_date,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,reason_code_budget,reason_code_competitor,reason_code_features,reason_code_pricing,...,closed_at_min,closed_at_max,resolution_time_hours_mean,first_response_time_minutes_mean,satisfaction_score_mean,escalation_flag_sum,priority_urgent_sum,priority_high_sum,priority_low_sum,priority_medium_sum
count,500.000000,352,352.000000,352.000000,352.000000,352.000000,352.000000,352.000000,352.000000,352.000000,...,492,492,492.000000,492.000000,466.000000,492.000000,492.000000,492.000000,492.000000,492.000000
mean,20.560000,2024-08-19 05:10:54.545454592,24.580256,0.349432,0.150568,0.173295,0.295455,0.261364,0.323864,0.258523,...,2023-06-26 22:10:36.585366016,2024-07-27 23:45:29.268292352,36.238515,88.560094,3.964163,0.193089,1.044715,1.036585,0.985772,0.997967
min,1.000000,2023-03-11 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2023-01-03 03:00:00,2023-01-31 13:00:00,1.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.000000,2024-06-17 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2023-03-01 08:00:00,2024-05-08 06:30:00,29.000000,71.333333,3.500000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,15.000000,2024-10-03 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2023-05-15 06:30:00,2024-09-09 07:00:00,36.000000,88.291667,4.000000,0.000000,1.000000,1.000000,1.000000,1.000000
75%,28.000000,2024-12-04 00:00:00,19.797500,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,...,2023-08-30 00:30:00,2024-11-16 18:30:00,43.000000,106.250000,4.333333,0.000000,2.000000,2.000000,2.000000,2.000000
max,163.000000,2024-12-31 00:00:00,444.100000,2.000000,2.000000,2.000000,3.000000,3.000000,2.000000,2.000000,...,2024-10-03 03:00:00,2024-12-31 19:00:00,72.000000,175.000000,5.000000,3.000000,6.000000,5.000000,5.000000,4.000000
std,21.044718,NaN,53.309174,0.559863,0.388656,0.421735,0.520992,0.522853,0.525909,0.469814,...,NaN,NaN,11.823256,27.942216,0.570420,0.420107,1.037991,0.971426,0.993769,0.971066


In [512]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 95 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   account_id                            500 non-null    object        
 1   account_name                          500 non-null    object        
 2   industry                              500 non-null    object        
 3   country                               500 non-null    object        
 4   signup_date                           500 non-null    object        
 5   referral_source                       500 non-null    object        
 6   plan_tier                             500 non-null    object        
 7   seats_x                               500 non-null    int64         
 8   is_trial_current                      500 non-null    bool          
 9   churn_flag                            500 non-null    bool          
 10  ch

In [513]:
final_df['signup_date'] = pd.to_datetime(final_df['signup_date'])


In [514]:
final_df['signup_date']

,signup_date
0,2024-10-16
1,2023-08-17
2,2024-08-27
3,2023-08-27
4,2024-10-27
...,...
495,2024-06-28
496,2023-12-21
497,2024-11-07
498,2023-07-31


In [515]:
final_df['account_age_days'] = (final_df['usage_date'] - final_df['signup_date']).dt.days

In [516]:
final_df['subscription_duration_days'] = (final_df['end_date'] - final_df['start_date']).dt.days

In [517]:
final_df['ticket_span_days'] = (final_df['closed_at_max'] - final_df['submitted_at_min']).dt.days

In [518]:
observation_date = final_df['usage_date'].max()
final_df['days_since_last_usage'] = (observation_date - final_df['usage_date']).dt.days

In [519]:
final_df['avg_usage_time'] = final_df['usage_duration_secs'] / final_df['usage_count']

In [520]:
final_df['revenue_per_seat'] = final_df['arr_amount'] / final_df['seats_x']

In [521]:
final_df['support_flag'] = (final_df['resolution_time_hours_mean'] > 0).astype(int)

In [522]:
final_df['account_age_days'] = (final_df['usage_date'] - final_df['signup_date']).dt.days

In [523]:
final_df['support_pressure'] = (
    final_df['priority_urgent_sum'] * 4 +
    final_df['priority_high_sum'] * 3 +
    final_df['priority_medium_sum'] * 2 +
    final_df['priority_low_sum']
)

In [524]:
final_df['plan_momentum'] = final_df['upgrade_flag'] - final_df['downgrade_flag']

In [525]:
final_df['support_risk'] = 5 - final_df['satisfaction_score_mean']

In [526]:
final_df['referral_source'].value_counts()

,count
referral_source,
organic,114
other,103
ads,98
event,96
partner,89


In [527]:
final_df.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 105 columns):
 #    Column                                Non-Null Count  Dtype         
---   ------                                --------------  -----         
 0    account_id                            500 non-null    object        
 1    account_name                          500 non-null    object        
 2    industry                              500 non-null    object        
 3    country                               500 non-null    object        
 4    signup_date                           500 non-null    datetime64[ns]
 5    referral_source                       500 non-null    object        
 6    plan_tier                             500 non-null    object        
 7    seats_x                               500 non-null    int64         
 8    is_trial_current                      500 non-null    bool          
 9    churn_flag                            500 non-null    bool     

In [528]:
feature_cols = [col for col in final_df.columns if "feature_name_" in col]
final_df['feature_adoption_score'] = final_df[feature_cols].sum(axis=1)

In [529]:
final_df['features_used_count'] = (final_df[feature_cols] > 0).sum(axis=1)

In [530]:
leakage_cols = [
'churn_date',
'refund_amount_usd',
'preceding_upgrade_flag',
'preceding_downgrade_flag',
'is_reactivation',
'reason_code_budget',
'reason_code_competitor',
'reason_code_features',
'reason_code_pricing',
'reason_code_support',
'reason_code_unknown',
'feedback_text_missing_features',
'feedback_text_switched_to_competitor',
'feedback_text_too_expensive'
]
IDs = ['account_id', 'account_name']
raw_dates = [
'signup_date',
'start_date',
'end_date',
'usage_date',
'submitted_at_min',
'submitted_at_max',
'closed_at_min',
'closed_at_max'
]
features_to_remove = leakage_cols + IDs + raw_dates + feature_cols
final_df.drop(columns=features_to_remove, inplace=True)



In [531]:
final_df

,industry,country,referral_source,plan_tier,seats_x,is_trial_current,churn_flag,arr_amount,auto_renew_flag,billing_frequency_annual,...,ticket_span_days,days_since_last_usage,avg_usage_time,revenue_per_seat,support_flag,support_pressure,plan_momentum,support_risk,feature_adoption_score,features_used_count
0,EdTech,US,partner,Basic,9,False,False,151236,8,3,...,458.0,21,284.745794,16804.000000,1,7.0,1,2.000000,55.0,27
1,FinTech,IN,other,Basic,18,False,True,120048,6,6,...,529.0,2,284.890141,6669.333333,1,10.0,3,1.000000,35.0,23
2,DevTools,US,organic,Basic,1,False,False,219432,15,7,...,502.0,9,305.980512,219432.000000,1,8.0,0,0.333333,83.0,34
3,HealthTech,UK,other,Basic,24,True,False,111300,5,3,...,369.0,20,268.397906,4637.500000,1,4.0,2,NaN,41.0,26
4,HealthTech,US,event,Enterprise,35,False,True,585132,6,5,...,504.0,8,372.675302,16718.057143,1,20.0,2,1.200000,58.0,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,DevTools,CA,ads,Pro,9,False,False,53364,8,4,...,368.0,7,323.985075,5929.333333,1,14.0,0,1.500000,39.0,26
496,FinTech,US,organic,Basic,9,False,False,145644,8,6,...,503.0,0,306.663866,16182.666667,1,15.0,3,0.000000,53.0,31
497,DevTools,UK,organic,Basic,9,False,True,199356,7,4,...,31.0,22,265.019651,22150.666667,1,6.0,-1,1.000000,46.0,24
498,EdTech,US,organic,Pro,18,False,False,232584,11,5,...,500.0,19,315.334501,12921.333333,1,10.0,1,1.250000,55.0,32


In [532]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 43 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   industry                          500 non-null    object 
 1   country                           500 non-null    object 
 2   referral_source                   500 non-null    object 
 3   plan_tier                         500 non-null    object 
 4   seats_x                           500 non-null    int64  
 5   is_trial_current                  500 non-null    bool   
 6   churn_flag                        500 non-null    bool   
 7   arr_amount                        500 non-null    int64  
 8   auto_renew_flag                   500 non-null    int64  
 9   billing_frequency_annual          500 non-null    int64  
 10  billing_frequency_monthly         500 non-null    int64  
 11  past_churn_count                  500 non-null    int64  
 12  downgrad

In [533]:
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded = ohe.fit_transform(
    final_df[['industry','country','referral_source']]
)
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(['industry','country','referral_source']),
    index=final_df.index
)
final_df = pd.concat([final_df, encoded_df], axis=1)

In [534]:
os = OrdinalEncoder()
final_df['plan_tier_encoded'] = os.fit_transform(final_df[['plan_tier']])

In [535]:
final_df.drop(columns=final_df[['industry','country','referral_source','plan_tier']], inplace=True)

In [543]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 54 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   seats_x                           500 non-null    int64  
 1   is_trial_current                  500 non-null    bool   
 2   churn_flag                        500 non-null    bool   
 3   arr_amount                        500 non-null    int64  
 4   auto_renew_flag                   500 non-null    int64  
 5   billing_frequency_annual          500 non-null    int64  
 6   billing_frequency_monthly         500 non-null    int64  
 7   past_churn_count                  500 non-null    int64  
 8   downgrade_flag                    500 non-null    int64  
 9   error_count                       500 non-null    float64
 10  is_beta_feature                   500 non-null    float64
 11  trial_count                       500 non-null    int64  
 12  mrr_amou

In [542]:
support_cols = [
    'resolution_time_hours_mean',
    'first_response_time_minutes_mean',
    'escalation_flag_sum',
    'priority_urgent_sum',
    'priority_high_sum',
    'priority_low_sum',
    'priority_medium_sum',
    'support_pressure',
    'ticket_span_days',
    'support_risk'
]
final_df[support_cols] = final_df[support_cols].fillna(0)

final_df['satisfaction_score_mean'] = final_df['satisfaction_score_mean'].fillna(final_df['satisfaction_score_mean'].median())

In [539]:
final_df['subscription_duration_days'] = final_df['subscription_duration_days'].fillna(final_df['account_age_days'])

In [541]:
final_df['support_risk']

,support_risk
0,2.000000
1,1.000000
2,0.333333
3,NaN
4,1.200000
...,...
495,1.500000
496,0.000000
497,1.000000
498,1.250000
